# 00 — Euclidean Distance (The Wrong but Useful Way)

The distance formula from algebra class works perfectly in two-dimensional Cartesian space. Latitude and longitude look like a two-dimensional Cartesian space. This notebook is about what happens when you apply the former to the latter — and why the result is wrong, how wrong it is, and when you can get away with it anyway.

## 1. The Distance Formula

Given two points in a flat plane, the straight-line distance between them is:

```text
d = √( (x₂ - x₁)² + (y₂ - y₁)² )
```

This is the Pythagorean theorem applied to the right triangle formed by the two points and their shared corner. It works because the axes are in consistent, uniform units — both in meters, both in pixels, both in whatever.

Mapping this onto coordinates:

```text
x  →  longitude
y  →  latitude
```

The formula becomes:

```python
import math
d = math.sqrt((lon2 - lon1)**2 + (lat2 - lat1)**2)
```

And you get a number. The question — which this notebook will make you earn — is: **a number in what units?**

## 2. Compute Distance Between Two Points

In [1]:
import math

# Two points in [lon, lat] order — GeoJSON convention
p1 = (-98.5, 33.8)   # near Wichita Falls, TX
p2 = (-97.2, 34.1)   # about 130 km to the northeast

dx = p2[0] - p1[0]   # longitude difference
dy = p2[1] - p1[1]   # latitude difference

d = math.sqrt(dx**2 + dy**2)

print(f"p1: {p1}")
print(f"p2: {p2}")
print(f"dx (Δ lon): {dx}")
print(f"dy (Δ lat): {dy}")
print(f"d:  {d:.4f}")

p1: (-98.5, 33.8)
p2: (-97.2, 34.1)
dx (Δ lon): 1.2999999999999972
dy (Δ lat): 0.30000000000000426
d:  1.3342


## 3. What Are the Units?

You just computed `d ≈ 1.334`.

**1.334 what?**

Not kilometers. Not miles. Not meters.

**Degrees.**

`dx` and `dy` are differences in degrees of longitude and latitude. Squaring them, adding them, and taking the square root gives you a number in degrees — a hybrid angular unit that combines east-west and north-south angle into a single diagonal measurement.

That is not a physically meaningful distance. A degree of longitude at 33°N covers about 93 km. A degree of latitude covers about 111 km. They are different lengths. Treating them as equivalent axes — which is exactly what the Euclidean formula does — introduces an immediate geometric error.

The actual distance between these two points is roughly **133 km**. The Euclidean formula gave `1.334` — which is off by a factor of ~100, and the factor depends entirely on where on Earth you are.

In [2]:
# Wrap it as a reusable function
def euclidean_distance(p1, p2):
    """
    Straight-line distance between two [lon, lat] points.
    Returns a value in degrees — NOT kilometers.
    Use only for relative comparisons within small, consistent areas.
    """
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    return math.sqrt(dx**2 + dy**2)


print(f"Euclidean distance: {euclidean_distance(p1, p2):.4f}°")
print("(Actual distance is ~133 km — the degree value is not comparable to km)")

Euclidean distance: 1.3342°
(Actual distance is ~133 km — the degree value is not comparable to km)


## 4. Visualize on a Map

Plotting both points with a line between them makes the geometry concrete. It also makes it obvious that the "distance" is a diagonal across the coordinate grid — not a physical path across the ground.

In [3]:
from ipyleaflet import Map, GeoJSON

points_fc = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": list(p1)}, "properties": {"name": "p1"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": list(p2)}, "properties": {"name": "p2"}},
    ]
}

line_fc = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "geometry": {"type": "LineString", "coordinates": [list(p1), list(p2)]},
        "properties": {"label": f"d = {euclidean_distance(p1, p2):.4f}°  (not km)"}
    }]
}

center_lat = (p1[1] + p2[1]) / 2
center_lon = (p1[0] + p2[0]) / 2

m = Map(center=(center_lat, center_lon), zoom=9)
m.add(GeoJSON(data=points_fc))
m.add(GeoJSON(data=line_fc, style={"color": "#e63946", "weight": 2}))
m

Map(center=[33.95, -97.85], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

## 5. When This Works

Euclidean distance on raw degrees is not entirely useless. Within a small, geographically consistent area, the degree-to-km ratio is approximately constant across all the points in your dataset — so the *relative ordering* of distances is roughly preserved even if the *absolute values* are wrong.

**Acceptable use cases:**
- Finding the nearest point out of a local set (same city, same county)
- Rough filtering — "which features are in the same neighborhood?"
- Sorting by proximity when you only care about rank, not exact distance
- Quick checks during development before switching to proper formulas

**Not acceptable:**
- Reporting an actual distance to a user
- Comparing distances across different latitudes
- Any computation where someone acts on the result
- Navigation, guidance, or range estimation of any kind

The test: *does the answer matter?* If yes, use Haversine. If you just need "roughly which direction is closer," Euclidean on degrees can hold the line temporarily.

---

## Exercise A — Distances Between Several Points

Compute the Euclidean distance from the base point to each of the targets below. Print the results sorted nearest to farthest.

In [4]:
import math

base = (-98.47, 33.91)

targets = [
    ("Tinker AFB",         (-97.37, 35.39)),
    ("NAS Fort Worth JRB", (-97.04, 32.85)),
    ("Lubbock",            (-101.87, 33.57)),
    ("Oklahoma City",      (-97.52, 35.47)),
    ("Abilene",            (-99.73, 32.45)),
]

def euclidean_degrees(a, b):
    lon1, lat1 = a
    lon2, lat2 = b
    return math.sqrt((lon1 - lon2)**2 + (lat1 - lat2)**2)

# compute distances
distances = []
for name, coords in targets:
    d = euclidean_degrees(base, coords)
    distances.append((name, d))

# sort nearest → farthest
distances_sorted = sorted(distances, key=lambda x: x[1])

# print results
for name, d in distances_sorted:
    print(f"{name:20s}  {d:.4f}")



NAS Fort Worth JRB    1.7800
Oklahoma City         1.8265
Tinker AFB            1.8440
Abilene               1.9285
Lubbock               3.4170


## Exercise B — Rank Nearest Points

Using `euclidean_distance`, write a function `nearest_n(base, points, n)` that returns the `n` closest points from a list, sorted by distance. Test it on the targets above.

In [5]:
import math

def nearest_n(base, named_points, n):
    """
    Returns the n closest (name, coord) pairs from named_points to base,
    sorted nearest first.
    """
    # compute (name, coord, distance)
    distances = [
        (name, coord, euclidean_distance(base, coord))
        for name, coord in named_points
    ]

    # sort by distance
    distances_sorted = sorted(distances, key=lambda x: x[2])

    # return only (name, coord) for the top n
    return [(name, coord) for name, coord, d in distances_sorted[:n]]


    pass


top3 = nearest_n(base, targets, 3)
for name, coord in top3:
    print(f"{name}: {euclidean_distance(base, coord):.4f}°")

NAS Fort Worth JRB: 1.7800°
Oklahoma City: 1.8265°
Tinker AFB: 1.8440°


## Exercise C — Visualize All Distances

Plot `base` and all five targets on a map. Draw a line from `base` to each target. Label each line with the Euclidean distance value (in the `properties` dict — you can inspect it in the browser dev tools or just print it before displaying).

In [6]:
from ipyleaflet import Map, Marker, GeoJSON, Polyline
from ipywidgets import VBox
import math

# Euclidean degree-space distance
def euclidean_distance(a, b):
    lon1, lat1 = a
    lon2, lat2 = b
    return math.sqrt((lon1 - lon2)**2 + (lat1 - lat2)**2)

# Base and targets
base = (-98.47, 33.91)

targets = [
    ("Tinker AFB",         (-97.37, 35.39)),
    ("NAS Fort Worth JRB", (-97.04, 32.85)),
    ("Lubbock",            (-101.87, 33.57)),
    ("Oklahoma City",      (-97.52, 35.47)),
    ("Abilene",            (-99.73, 32.45)),
]

# Create map
m = Map(center=[33.9, -98.4], zoom=6)

# Add base marker
base_marker = Marker(location=(base[1], base[0]), title="Base")
m.add(base_marker)

# Draw lines + markers + distance labels
for name, coord in targets:
    lon, lat = coord

    # Add target marker
    marker = Marker(location=(lat, lon), title=name)
    m.add(marker)

    # Compute Euclidean distance
    d = euclidean_distance(base, coord)

    # Build GeoJSON line with distance in properties
    line_feature = {
        "type": "Feature",
        "geometry": {
            "type": "LineString",
            "coordinates": [
                [base[0], base[1]],
                [lon, lat]
            ]
        },
        "properties": {
            "name": name,
            "euclidean_deg": d
        }
    }

    # Add line layer
    line_layer = GeoJSON(
        data=line_feature,
        style={"color": "blue", "weight": 2}
    )
    m.add(line_layer)

    # Optional: print distances so you can see them directly
    print(f"{name}: {d:.4f}°")

VBox([m])



Tinker AFB: 1.8440°
NAS Fort Worth JRB: 1.7800°
Lubbock: 3.4170°
Oklahoma City: 1.8265°
Abilene: 1.9285°


---

## Check Your Understanding

You compute Euclidean distances from a base in Texas to two targets:

```python
base     = (-98.5, 33.8)
target_a = (-97.5, 33.8)   # 1.0° to the east, same latitude
target_b = (-98.5, 34.8)   # 1.0° to the north, same longitude
```

Both return `euclidean_distance = 1.0000`.

**Question:** Are these two targets actually the same distance from the base in the real world? If not, which is farther — and why?

```python
No they are not the same real-world distance from the base, even though the euclidean degree math says "1.0000" for both 

The point 1 degree north (target B) is farther form the base than the point 1 degree east (target A)

Because:
one degree of latitude is always ~111km everywhere on Earth
One degree of longitude shrinks with latitude - at 34 degree N (northern Texas), it's only abiut 92-96km, not 111km.
```


---

## Next

In [01 — Haversine Distance](./01-Haversine_Distance.ipynb), we replace the Euclidean formula with one that accounts for the curvature of the Earth — producing real distances in kilometers that hold up anywhere on the globe.